# 03 — Exploratory Data Analysis & Risk Visualization
**Project:** Axlero Solutions — Credit Risk Scoring and Loan Default Analysis  
**Contributor:** Srirami Reddy Ailuri (Member 2) — Validated & Integrated

## 1. Overview & Key Findings
- **Dataset:** `data/processed/final_validated_dataset.csv` (`32,416` loans, `0` missing values, `0` duplicates).
- **Overall Default Rate:** **21.87% (21.9%)** (`7,089` defaults vs `25,327` repaid loans — 1 default per 3.57 repaid loans).
- **Important Methodological Note:** Observational patterns represent statistical **associations**, not causal mechanisms. Furthermore, `loan_grade` and `loan_int_rate` are assigned by the lender during underwriting and therefore partly reflect the lender's prior risk assessment.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load_data import load_validated_data
from src.visualization.eda_plots import generate_eda_figures

df = load_validated_data()
fig_dir = PROJECT_ROOT / "reports" / "figures"
eda_tables = generate_eda_figures(df, fig_dir)
print(f"Validated overall default rate: {eda_tables['overall_default_rate_pct']:.2f}% (n = {len(df):,})")

findfont: Failed to find font weight medium for DejaVu Sans, now using 400.


Validated overall default rate: 21.87% (n = 32,416)


## 2. Univariate & Bivariate Default Rate Summaries Across All 9 Core Figures

In [2]:
for title, key in [
    ("Figure 2 — Loan Grade Default Rates", "grade_table"),
    ("Figure 3 — Loan Burden (loan_percent_income) Default Rates", "burden_table"),
    ("Figure 4a — Prior Default on File", "prior_default_table"),
    ("Figure 4b — Home Ownership Status", "home_ownership_table"),
    ("Figure 5 — Loan Purpose (Intent)", "intent_table"),
    ("Figure 6 — Annual Income Quartiles", "income_quartile_table"),
    ("Figure 7a — Loan Principal Amount Bands", "loan_amount_table"),
    ("Figure 7b — Interest Rate Bands", "interest_rate_table"),
    ("Figure 8a — Applicant Age Bands", "age_table"),
    ("Figure 8b — Employment Length Bands", "emp_length_table"),
]:
    print(f"\n=== {title} ===")
    display(pd.DataFrame(eda_tables[key]).T)


=== Figure 2 — Loan Grade Default Rates ===


,rate,count
A,9.96,10703.0
B,16.32,10387.0
C,20.75,6438.0
D,59.06,3620.0
E,64.49,963.0
F,70.54,241.0
G,98.44,64.0



=== Figure 3 — Loan Burden (loan_percent_income) Default Rates ===


,rate,count
<= 10%,11.75,10430.0
10-20%,15.13,11999.0
20-30%,22.02,6168.0
> 30%,70.44,3819.0



=== Figure 4a — Prior Default on File ===


,rate,count
No Prior Default (N),18.43,26686.0
Prior Default (Y),37.87,5730.0


... (10 outputs left out)


=== Figure 8a — Applicant Age Bands ===


,rate,count
20-24,23.34,12222.0
25-29,21.40,11147.0
30-39,20.24,7291.0
40+,21.30,1756.0



=== Figure 8b — Employment Length Bands ===


,rate,count
0-1 yrs,27.87,6983.0
2-4 yrs,23.33,11026.0
5-9 yrs,18.34,10799.0
10+ yrs,16.38,3608.0


## 3. Multi-Factor Interaction Check & Remaining Extreme Values Audit
As documented by Member 2:
- Combining **bottom-quartile income (`<= 38.5k`)** with **high loan burden (`> 30%`)** produces a **77.8% default rate**, illustrating strong non-linear interaction effects.
- Within subprime grades (`D-G`, overall default rate `61.2%`), prior default on file adds very little incremental separation (`60.8%` vs `61.6%`) because historical default was already priced into the grade.

In [3]:
df["low_inc"] = df["person_income"] <= df["person_income"].quantile(0.25)
df["high_burden"] = df["loan_percent_income"] > 0.30
interaction_tbl = df.groupby(["low_inc", "high_burden"])["loan_status"].agg(
    count="count", default_rate_pct=lambda s: round(s.mean() * 100, 2)
)
print("Interaction of Low Income (Q1) x High Debt Burden (>30%):")
display(interaction_tbl)

Interaction of Low Income (Q1) x High Debt Burden (>30%):


count  default_rate_pct
low_inc high_burden                         
False   False        22436             11.99
        True          1876             62.85
True    False         6161             27.76
        True          1943             77.77